In [1]:
# Importing Libraries
import torch
import os
import copy
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from PIL import Image, UnidentifiedImageError  # FIX: removed duplicate PIL imports
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn.functional as F

In [2]:
#Check If GPU is available

print(torch.cuda.is_available())      
print(torch.cuda.get_device_name(0))  

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
#Various Transforms for our data


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.RandomResizedCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms defined.')


Transforms defined.


In [4]:
#Loading The Dataset
DATA_DIR = r"C:\Users\King Taylor #8\Desktop\Herb Identification Model\dataset"

# # FIX: define and run clean_dataset BEFORE loading ImageFolder
# # so corrupt files are removed before they get indexed
# def clean_dataset(root_dir):
#     removed = 0
#     corrupted_files = []

#     for split in ['train', 'val', 'test']:
#         split_path = os.path.join(root_dir, split)
#         for class_name in os.listdir(split_path):
#             class_path = os.path.join(split_path, class_name)
#             if not os.path.isdir(class_path):
#                 continue
#             for img_file in os.listdir(class_path):
#                 img_path = os.path.join(class_path, img_file)
#                 try:
#                     with Image.open(img_path) as img:
#                         img.load()  # fully decodes the image, catches truncated files
#                 except (OSError, UnidentifiedImageError, Exception):
#                     corrupted_files.append(img_path)
#                     os.remove(img_path)
#                     removed += 1

#     print(f"Scan complete. {removed} corrupted images removed.")
#     for f in corrupted_files:
#         print(f"  Removed: {f}")

# clean_dataset(DATA_DIR)

# Now safe to load — all corrupt images already removed
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  transform=val_transforms)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f'Classes ({num_classes}): {class_names}')
print(f'Train: {len(train_dataset)} images')
print(f'Val:   {len(val_dataset)} images')
print(f'Test:  {len(test_dataset)} images')

Classes (501): ['Abelmoschus Esculentus', 'Abelmoschus moschatus', 'Abrus precatorius', 'Abutilon mauritianum', 'Acacia ataxacantha', 'Acacia gourmaensis', 'Acacia nilotica', 'Acacia seyal', 'Acalypha indica', 'Acalypha ornata', 'Acalypha wilkesiana', 'Acanthospermum hispidum', 'Acanthus montanus', 'Achyranthes aspera', 'Acokanthera schimperi', 'Aconitum napellus', 'Acorus calamus', 'Adansonia digitata', 'Adhatoda vasica', 'Aerva lanata', 'Aframomum daniellii', 'Afzelia africana', 'Afzelia quanzensis', 'Agathosma betulina', 'Agave', 'Ageratum conyzoides', 'Agrimonia eupatoria', 'Albizia adianthifolia', 'Albizia anthelmintica', 'Albizia lebbeck', 'Albizia zygia', 'Alchornea cordifolia', 'Alchornea laxiflora', 'Allium ascalonicum', 'Allium cepa', 'Allium sativum', 'Allium ursinum', 'Allophylus africanus', 'Aloe buettneri', 'Aloe ferox', 'Aloe vera', 'Alpinia galanga', 'Alpinia officinarum', 'Alstonia boonei', 'Alstonia scholaris', 'Alternanthera pungens', 'Alternanthera sessilis', 'Amara

In [5]:
# NOTE: clean_dataset has been moved to run BEFORE ImageFolder loading (cell above).
# This cell is intentionally left empty to preserve cell numbering.

In [6]:
# Dealing With Class Imbalances Using Weighted Random Sampler
# FIX: use train_dataset.targets instead of iterating — zero image loads
labels = torch.tensor(train_dataset.targets)
class_counts = torch.bincount(labels)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
print("Sample weights computed.")

Sample weights computed.


In [7]:
# Initializing The Data Loaders
# NUM_WORKERS = 0
# BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    pin_memory=True,
    num_workers=8,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True,
    num_workers=8,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True,
    num_workers=8,
)

#print(f"DataLoaders ready. batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

In [8]:
# Load the Pretrained model for transfer learning
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
torch.backends.cudnn.benchmark = True
# Load pretrained EfficientNet-B4
weights = EfficientNet_B4_Weights.IMAGENET1K_V1
model = efficientnet_b4(weights=weights)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# FIX: use modern GradScaler API (torch.cuda.amp.autocast is deprecated in PyTorch 2.x)
scaler = torch.cuda.amp.GradScaler()

print("All layers frozen.")

All layers frozen.


C:\Users\King Taylor #8\AppData\Local\Temp\ipykernel_21064\2066438568.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
# Replace the classifier head (The Ending Part Of the Network For MultiClass Classification)
in_features = model.classifier[1].in_features  # 1792 for B4

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

# Move model to GPU
device = torch.device('cuda')
model = model.to(device)

print(f"Classifier head replaced. Output classes: {num_classes}")
print(model.classifier)


Classifier head replaced. Output classes: 501
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1792, out_features=512, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=512, out_features=501, bias=True)
)


In [10]:
# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer - only classifier head parameters are trainable (backbone is frozen)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

# Learning Rate Scheduler
# FIX: verbose parameter removed — deprecated in PyTorch 2.x, raises warning
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

print("Loss function, optimizer and scheduler ready.")

Loss function, optimizer and scheduler ready.


In [11]:
# # Training Loop 1

# def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=20):
#     best_model_wts = copy.deepcopy(model.state_dict())
#     best_acc = 0.0
#     history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

#     for epoch in range(num_epochs):
#         print(f'\nEpoch {epoch+1}/{num_epochs}')
#         print('-' * 30)

#         # --- Training Phase ---
#         model.train()
#         running_loss = 0.0
#         running_corrects = 0

#         train_bar = tqdm(train_loader, desc='Training', leave=True)
#         for inputs, labels in train_bar:
#             inputs = inputs.to(device)
#             labels = labels.to(device)

#             optimizer.zero_grad()
#             # FIX: use modern torch.autocast API
#             with torch.autocast('cuda'):
#                 outputs = model(inputs)
#                 loss = criterion(outputs, labels)
#             scaler.scale(loss).backward()
#             scaler.step(optimizer)
#             scaler.update()

#             _, preds = torch.max(outputs, 1)
#             running_loss += loss.item() * inputs.size(0)
#             running_corrects += torch.sum(preds == labels.data)

#             train_bar.set_postfix(loss=f'{loss.item():.4f}')

#         train_loss = running_loss / len(train_dataset)
#         train_acc = running_corrects.double() / len(train_dataset)

#         # --- Validation Phase ---
#         model.eval()
#         running_loss = 0.0
#         running_corrects = 0

#         val_bar = tqdm(val_loader, desc='Validating', leave=True)
#         with torch.no_grad():
#             for inputs, labels in val_bar:
#                 inputs = inputs.to(device)
#                 labels = labels.to(device)

#                 # FIX: use modern torch.autocast API
#                 with torch.autocast('cuda'):
#                     outputs = model(inputs)
#                     loss = criterion(outputs, labels)

#                 _, preds = torch.max(outputs, 1)
#                 running_loss += loss.item() * inputs.size(0)
#                 running_corrects += torch.sum(preds == labels.data)

#                 val_bar.set_postfix(loss=f'{loss.item():.4f}')

#         val_loss = running_loss / len(val_dataset)
#         val_acc = running_corrects.double() / len(val_dataset)

#         scheduler.step(val_acc)

#         if val_acc > best_acc:
#             best_acc = val_acc
#             best_model_wts = copy.deepcopy(model.state_dict())
#             torch.save(model.state_dict(), 'best_model.pth')
#             print(f'  New best model saved! Val Acc: {val_acc:.4f}')

#         history['train_loss'].append(train_loss)
#         history['train_acc'].append(train_acc.item())
#         history['val_loss'].append(val_loss)
#         history['val_acc'].append(val_acc.item())

#         print(f'  Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}')
#         print(f'  Val Loss:   {val_loss:.4f}  Val Acc:   {val_acc:.4f}')

#     model.load_state_dict(best_model_wts)
#     print(f'\nTraining complete. Best Val Acc: {best_acc:.4f}')
#     return model, history


# model, history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=20)

In [12]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    # Unfreeze the last 3 blocks of the backbone for Phase 2 Training
for param in model.features[-3:].parameters():
    param.requires_grad = True

# Confirm what's frozen and what's not
for i, block in enumerate(model.features):
    frozen = not any(p.requires_grad for p in block.parameters())
    print(f"Block {i}: {'Frozen' if frozen else 'Unfrozen'}")

Block 0: Frozen
Block 1: Frozen
Block 2: Frozen
Block 3: Frozen
Block 4: Frozen
Block 5: Frozen
Block 6: Unfrozen
Block 7: Unfrozen
Block 8: Unfrozen


In [13]:
# Phase 2 Optimizer - different learning rates for different parts
optimizer_phase2 = optim.Adam([
    {'params': model.features[-3:].parameters(), 'lr': 1e-5},  # unfrozen backbone blocks
    {'params': model.classifier.parameters(), 'lr': 1e-4}       # classifier head
], weight_decay=1e-4)

scheduler_phase2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase2, 
    T_max=20,
    eta_min=1e-7
)

#The Loss Function
criterion = nn.CrossEntropyLoss()

print("Phase 2 optimizer ready.")

Phase 2 optimizer ready.


In [14]:
# # Phase 2 Training Loop
# num_epochs_phase2 = 20
# best_val_acc = 0.0
# best_model_wts = copy.deepcopy(model.state_dict())

# for epoch in range(num_epochs_phase2):

#     model.train()
#     running_loss = 0.0
#     running_corrects = 0

#     train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs_phase2} [Train]", leave=False)
#     for inputs, labels in train_bar:
#         inputs = inputs.to(device)
#         labels = labels.to(device)

#         optimizer_phase2.zero_grad()
#         # FIX: use modern torch.autocast API
#         with torch.autocast('cuda'):
#             outputs = model(inputs)
#             loss = criterion(outputs, labels)
#         scaler.scale(loss).backward()
#         scaler.step(optimizer_phase2)
#         scaler.update()

#         _, preds = torch.max(outputs, 1)
#         running_loss += loss.item() * inputs.size(0)
#         running_corrects += torch.sum(preds == labels.data)

#         train_bar.set_postfix(loss=f"{loss.item():.4f}")

#     scheduler_phase2.step()

#     train_loss = running_loss / len(train_dataset)
#     train_acc  = running_corrects.double() / len(train_dataset)

#     # FOR VALIDATION
#     model.eval()
#     val_loss = 0.0
#     val_corrects = 0

#     val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs_phase2} [Val]  ", leave=False)
#     with torch.no_grad():
#         for inputs, labels in val_bar:
#             inputs = inputs.to(device)
#             labels = labels.to(device)

#             # FIX: use modern torch.autocast API
#             with torch.autocast('cuda'):
#                 outputs = model(inputs)
#                 loss = criterion(outputs, labels)

#             _, preds = torch.max(outputs, 1)
#             val_loss += loss.item() * inputs.size(0)
#             val_corrects += torch.sum(preds == labels.data)

#             val_bar.set_postfix(loss=f"{loss.item():.4f}")

#     val_loss = val_loss / len(val_dataset)
#     val_acc  = val_corrects.double() / len(val_dataset)

#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         best_model_wts = copy.deepcopy(model.state_dict())
#         torch.save(model.state_dict(), 'best_herb_model.pth')
#         print(f"  --> New best model saved! Val Acc: {val_acc:.4f}")

#     print(f"Epoch {epoch+1}/{num_epochs_phase2} "
#           f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} "
#           f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

# model.load_state_dict(best_model_wts)
# print(f"\nPhase 2 Complete. Best Val Acc: {best_val_acc:.4f}")

In [15]:
# Unfreeze blocks 4, 5, 6, 7, 8
for param in model.features[-5:].parameters():
    param.requires_grad = True

# Confirm what's frozen and what's not
for i, block in enumerate(model.features):
    frozen = not any(p.requires_grad for p in block.parameters())
    print(f"Block {i}: {'Frozen' if frozen else 'Unfrozen'}")

Block 0: Frozen
Block 1: Frozen
Block 2: Frozen
Block 3: Frozen
Block 4: Unfrozen
Block 5: Unfrozen
Block 6: Unfrozen
Block 7: Unfrozen
Block 8: Unfrozen


In [16]:
# Phase 3 Optimizer
optimizer_phase3 = optim.Adam([
    {'params': model.features[-5:].parameters(), 'lr': 5e-6},  # unfrozen backbone blocks
    {'params': model.classifier.parameters(), 'lr': 5e-5}       # classifier head
], weight_decay=1e-4)

scheduler_phase3 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase3,
    T_max=30,
    eta_min=1e-8
)
print("Phase 3 Optimizer Ready!! ")



Phase 3 Optimizer Ready!! 


In [18]:
# Clear memory   #I HAD TO EMPTY GPU MEMORY BECAUSE TRAINING WAS CRASHING DUE TO LIMITED MEMORY
torch.cuda.empty_cache()
import gc
gc.collect()

# Reload best Phase 2 weights
model.load_state_dict(torch.load(r"C:\Users\King Taylor #8\Desktop\Herb Identification Model\best_herb_model_phase3.pth", weights_only=True))
model = model.to(device)

print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
print(f"GPU memory cached: {torch.cuda.memory_reserved()/1024**2:.1f} MB")

GPU memory allocated: 72.4 MB
GPU memory cached: 152.0 MB


In [ ]:
# --- Phase 3 Training Loop ---
num_epochs_phase3 = 15
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
patience = 5
epochs_no_improve = 0

criterion = nn.CrossEntropyLoss()

for epoch in range(num_epochs_phase3):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs_phase3} [Train]", leave=False)
    for inputs, labels in train_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer_phase3.zero_grad()
        # FIX: use modern torch.autocast API
        with torch.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer_phase3)
        scaler.update()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler_phase3.step()

    # FIX: use len(train_dataset) as denominator — consistent with Phase 1 & 2
    # (len(train_loader) * batch_size overcounts when last batch is smaller)
    train_loss = running_loss / len(train_dataset)
    train_acc  = running_corrects.double() / len(train_dataset)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs_phase3} [Val]  ", leave=False)
    with torch.no_grad():
        for inputs, labels in val_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # FIX: use modern torch.autocast API
            with torch.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

            val_bar.set_postfix(loss=f"{loss.item():.4f}")

    # FIX: use len(val_dataset) for both val_loss and val_acc — consistent denominators
    val_loss = val_loss / len(val_dataset)
    val_acc  = val_corrects.double() / len(val_dataset)

    # --- Early Stopping & Save Best Model ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_herb_model_4.pth')
        epochs_no_improve = 0
        print(f"  --> New best model saved! Val Acc: {val_acc:.4f}")
    else:
        epochs_no_improve += 1
        print(f"  --> No improvement for {epochs_no_improve}/{patience} epochs")

    print(f"Epoch {epoch+1}/{num_epochs_phase3} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} "
          f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs!")
        break

model.load_state_dict(best_model_wts)
print(f"\nPhase 3 Complete. Best Val Acc: {best_val_acc:.4f}")

Epoch 1/15 [Train]:   0%|          | 0/8497 [00:00<?, ?it/s]

In [ ]:
# TO TEST THE MODEL
model.eval()
all_preds = []
all_labels = []

test_bar = tqdm(test_loader, desc="Testing", leave=False)
with torch.no_grad():
    for inputs, labels in test_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # FIX: add autocast for consistency with training and faster inference
        with torch.autocast('cuda'):
            outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# --- Overall Accuracy ---
test_acc = np.sum(all_preds == all_labels) / len(all_labels)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# --- Per Class Accuracy ---
print("\nPer Class Accuracy:")
for i, class_name in enumerate(class_names):
    class_mask = all_labels == i
    class_acc = np.sum(all_preds[class_mask] == all_labels[class_mask]) / np.sum(class_mask)
    print(f"{class_name}: {class_acc*100:.2f}%")

In [ ]:


# --- Compute Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)

# --- Plot ---
plt.figure(figsize=(40, 40))
sns.heatmap(
    cm,
    annot=False,        # too many classes for numbers to be readable
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5
)

plt.title('Confusion Matrix - Herb Identification Model', fontsize=24, pad=20)
plt.ylabel('True Label', fontsize=16)
plt.xlabel('Predicted Label', fontsize=16)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to confusion_matrix.png")

# --- Most Confused Pairs ---
print("\nTop 10 Most Confused Class Pairs:")
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)  # ignore correct predictions
confused_pairs = np.dstack(np.unravel_index(np.argsort(cm_no_diag.ravel())[::-1], cm_no_diag.shape))[0]

for i, (true_idx, pred_idx) in enumerate(confused_pairs[:10]):
    print(f"{i+1}. '{class_names[true_idx]}' predicted as '{class_names[pred_idx]}': {cm_no_diag[true_idx, pred_idx]} times")

In [ ]:
def predict_herb(image_path, model, class_names, device, top_k=5):
    # --- Load and preprocess image ---
    image = Image.open(image_path).convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((400, 400)),
        transforms.CenterCrop(380),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    input_tensor = transform(image).unsqueeze(0).to(device)

    # --- Run inference ---
    model.eval()
    with torch.no_grad():
        # FIX: use modern torch.autocast API
        with torch.autocast('cuda'):
            outputs = model(input_tensor)
        probabilities = F.softmax(outputs.float(), dim=1)

    # --- Get top K predictions ---
    top_probs, top_indices = torch.topk(probabilities, top_k)
    top_probs = top_probs.squeeze().cpu().numpy()
    top_indices = top_indices.squeeze().cpu().numpy()

    # --- Display results ---
    print(f"\nImage: {image_path}")
    print(f"\nTop {top_k} Predictions:")
    print("-" * 50)
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        print(f"{i+1}. {class_names[idx]:<50} {prob*100:.2f}%")

    # --- Show image ---
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Predicted: {class_names[top_indices[0]]}\nConfidence: {top_probs[0]*100:.2f}%")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return class_names[top_indices[0]], top_probs[0]

# --- Test it ---
image_path = r"C:\Users\King Taylor #8\Downloads\images.jpg"  # change this to your image path
predicted_class, confidence = predict_herb(image_path, model, class_names, device)